# Newsio+ — 在 Colab 上用 QLoRA 微調 Llama 3.2 3B

這個 notebook 帶你在免費的 Colab T4 GPU 上，微調 Llama 3.2 3B
做「英文 → 繁體中文（zh-TW）新聞翻譯」。內容和 repo 裡的 `train_lora.py` 一致，
只是拆成一格一格方便照著跑。

**開始前**：上方選單 → 「執行階段」→「變更執行階段類型」→ 硬體加速器選 **T4 GPU**。

課程網站：https://plus.newsio.io ｜ starter repo：https://github.com/wil-li-la/newsio-plus-starter

## 1. 安裝套件

大約需要 2–3 分鐘。裝完如果 Colab 提示要重新啟動 runtime，重啟後從下一格繼續即可。

In [ ]:
%pip install -q "torch>=2.4.0" "transformers>=4.46.0" "peft>=0.14.0" "datasets>=3.0.0" "accelerate>=1.1.0" "bitsandbytes>=0.45.0"

import torch
assert torch.cuda.is_available(), "沒偵測到 GPU！請到「執行階段 → 變更執行階段類型」選 T4 GPU。"
print("GPU:", torch.cuda.get_device_name(0))

## 2. 上傳訓練資料 train.jsonl

還沒有資料？到 https://plus.newsio.io/data.html 用 Google 登入，
`train.jsonl`（14,746 筆 EN→zh-TW 訓練配對）會寄到你的信箱。

執行下面這格，選擇你下載的 `train.jsonl` 上傳（約 23MB，需要一點時間）。

In [ ]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/train.jsonl"):
    uploaded = files.upload()  # 選擇 train.jsonl
    fname = next(iter(uploaded))
    os.rename(fname, "data/train.jsonl")
print("資料就緒：data/train.jsonl,", os.path.getsize("data/train.jsonl") // 1024, "KB")

## 3. 準備訓練資料

把每筆資料組成 Llama 3.2 的 chat 格式：

- **system**：固定的翻譯編輯指令
- **user**：英文標題 + 重點
- **assistant**：繁中標題 + 重點（模型要學的答案）

loss 只算在 assistant 的部分（prompt 的 label 遮成 -100）—
模型只需要學「怎麼翻」，不用學「怎麼出題」。

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"  # 不用申請權限的鏡像
MAX_LENGTH = 1024

# 固定的 system prompt — 訓練與推論都要用同一句。
SYSTEM_PROMPT = (
    "你是 Newsio 的新聞翻譯編輯。"
    "將英文新聞標題與重點翻譯成台灣慣用的繁體中文，保持 Newsio 簡潔風格。"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Llama 沒有 pad token，借用 eos


def as_lines(value):
    # en_bullets / zh_bullets 可能是 list 也可能是字串 — 一律轉成「- 重點」逐行文字。
    if isinstance(value, str):
        value = [value]
    return "\n".join(f"- {item}".strip() for item in value if str(item).strip())


def build_messages(record):
    user_text = record["en_title"].strip() + "\n" + as_lines(record["en_bullets"])
    assistant_text = record["zh_title"].strip() + "\n" + as_lines(record["zh_bullets"])
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]


def tokenize_record(record):
    messages = build_messages(record)
    # 完整對話（含 assistant 答案）
    # transformers 5.x 下 tokenize=True 會回傳 BatchEncoding（dict），要取 ['input_ids']
    full_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, return_dict=True)['input_ids']
    # 只有 prompt（system+user）— 用來知道要遮住幾個 token
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=True)['input_ids']

    full_ids = full_ids[:MAX_LENGTH]
    labels = list(full_ids)
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels[:prompt_len] = [-100] * prompt_len  # -100 = 不算 loss
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids),
            "labels": labels, "prompt_len": prompt_len}


records = []
with open("data/train.jsonl", encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            records.append(json.loads(line))
# 丟掉沒有繁中重點的資料（約 12%% 的原始資料 zh_bullets 是空的，留著會教模型不輸出重點）
records = [r for r in records if (r.get("zh_bullets") or []) and str(r.get("zh_title") or "").strip()]
print(f"共 {len(records):,} 筆訓練配對（已過濾空 zh_bullets）")

dataset = Dataset.from_list(records)
dataset = dataset.map(tokenize_record, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda ex: ex["prompt_len"] < len(ex["input_ids"]))
dataset = dataset.remove_columns(["prompt_len"])
dataset

## 4. 載入模型 + 掛上 LoRA adapter（r=16）

- 基底模型用 **4-bit（nf4）量化** 載入 → 3B 模型只吃約 2GB VRAM。
- LoRA 只掛在 attention 的 **q_proj / k_proj / v_proj / o_proj** 上，
  每個 transformer block 多 r × 20,480 個參數，28 個 block 合計 **9,175,040** 個
  可訓練參數 — 只佔整個模型的 **0.29%**。

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

RANK = 16  # LoRA rank（r）— 想實驗可以改 1~64；alpha 固定 2×r
use_bf16 = torch.cuda.is_bf16_supported()  # T4 只支援 fp16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.config.use_cache = False  # 訓練時關掉 KV cache
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=RANK * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 只動 q/k/v/o
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# get_nb_trainable_parameters() 回傳 (可訓練參數, 含 adapter 的總參數)；
# 減掉 trainable 才是基底模型的參數量（Llama 3.2 3B = 3,212,749,824）。
trainable, total_with_adapter = model.get_nb_trainable_parameters()
base_params = total_with_adapter - trainable
print(f"LoRA r={RANK}: 可訓練參數 {trainable:,}，基底模型 {base_params:,}（佔 {100*trainable/base_params:.2f}%）")

## 5. 訓練（1 epoch）

T4 上全部 14,746 筆約需 1.5–2.5 小時。趕時間的話，把下面的
`TRAIN_SAMPLES` 改成 2000 左右，20–30 分鐘就能看到效果。

In [ ]:
from transformers import Trainer, TrainingArguments

TRAIN_SAMPLES = None  # None = 全部；想快速實驗改成 2000

train_ds = dataset if TRAIN_SAMPLES is None else dataset.select(range(TRAIN_SAMPLES))


def pad_collator(features):
    # 把一個 batch 內長短不一的樣本補到等長（labels 用 -100 補，不影響 loss）。
    max_len = max(len(f["input_ids"]) for f in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for f in features:
        pad = max_len - len(f["input_ids"])
        batch["input_ids"].append(f["input_ids"] + [tokenizer.pad_token_id] * pad)
        batch["attention_mask"].append(f["attention_mask"] + [0] * pad)
        batch["labels"].append(f["labels"] + [-100] * pad)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}


training_args = TrainingArguments(
    output_dir="out/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # 有效 batch size = 2×8 = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_ds, data_collator=pad_collator)
trainer.train()

model.save_pretrained("out/adapter")
tokenizer.save_pretrained("out/adapter")
print("adapter 已存到 out/adapter/")

## 6. 試翻一則新聞

用剛訓練好的 adapter 翻一則範例新聞。也可以把 `title` / `bullets` 換成
今天 Newsio 上任何一則英文新聞試試看。

In [ ]:
title = "OpenAI releases GPT-5.2 with improved multilingual reasoning"
bullets = [
    "The new model scores 30% higher on non-English benchmarks.",
    "API pricing stays unchanged from GPT-5.1.",
    "Enterprise customers get early access starting next week.",
]

user_text = title + "\n" + "\n".join(f"- {b}" for b in bullets)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_text},
]

model.eval()
# transformers 5.x 下要用 return_dict=True 再取 ["input_ids"]
encoded = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
).to(model.device)
input_ids = encoded["input_ids"]

with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=512,
        do_sample=False,  # greedy decoding，結果穩定可重現
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )

print("=== 英文原文 ===")
print(user_text)
print()
print("=== 繁體中文翻譯 ===")
print(tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip())

## 7. 下載 adapter

打包 `out/adapter/`（只有幾十 MB — 這就是 LoRA 的好處）下載到自己的電腦。
之後在本機用 starter repo 的 `test_translate.py` 就能載入這個 adapter。

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("newsio_plus_adapter", "zip", "out/adapter")
files.download("newsio_plus_adapter.zip")

## 下一步

- 回到課程網站 https://plus.newsio.io 看 LoRA 的原理講解
- 試試不同的 rank（把第 4 格的 `RANK` 改成 4、32、64，比較翻譯品質與參數量）
- 在本機 clone starter repo，把 adapter zip 解壓到 `out/adapter/`，
  用 `python test_translate.py` 隨時翻新聞